# Cape Verde Is Bigger Than Cape Verde — the reproducible notebook

This notebook re-runs, **from the raw dataset**, the pipeline behind every headline
number in the blog *Cape Verde Is Bigger Than Cape Verde* (cape-verde-diaspora) and
**asserts** that each reproduced figure matches the published one. If every cell runs
clean, every number on the page is reproducible.

**How to run (local only — there is no Colab branch):** download this notebook next to
the dataset (or set `CAPEVERDE_DATA_DIR` to the data directory) and run all cells.
Requires Python 3 + pandas.

| Cell | Re-expresses | Backs |
|---|---|---|
| `cell_setup` | DATA_DIR resolver + guarded reads | — |
| `cell_diaspora` | `code/diaspora_analysis.py` | ana_01–ana_05 (146,396 floor; 52.5/23.8/76.3%; ×2.83/×7.15; São Tomé −73.8%; 13.8→27.9 per 100) |
| `cell_remittances` | `code/remittances_analysis.py` | ana_06–ana_08 (28.2→12.3% of GDP; #18 of 160; #2 of 90; 21 top-10 years) |
| `cell_squad` | `code/squad_analysis.py` | ana_09–ana_11 (15/26 = 57.7%; 414/823 = 50.3%; 33/75 = 44.0%; zero domestic) |
| `cell_elo` | `code/elo_ratings.py` | ana_12/14/15/16 (peak 1702 on 2026-06-26; #138→#64; gap-440 win = #73; Kosovo min-above) |
| `cell_outcome_model` | `code/fit_outcome_model.py` (published constants) | ana_19 (3.4/8.3/88.3, advance 7.6) |
| `cell_wc_population` | `code/wc_smallest_nations.py` | ana_17/18 (3rd-least-populous ever; 5.1×; 89×; 648×) |

The notebook re-expresses the `code/` scripts' logic; it does not import or mutate
them, and regenerated files land in `verify/_repro_out/` only. Data reads are
read-only. The ordered-logit **fit** (C=0.6614, B=2.0468, H=0.2965) is reproduced by
`code/fit_outcome_model.py` (a long-running optimisation); this notebook re-applies the
published constants to the reproduced Elo ratings and asserts the published
probabilities — the constants' provenance is the script plus its printed fit log.


In [ ]:
# cell_setup — imports, DATA_DIR resolver, guarded reads, REPRO_OUT
import os
from pathlib import Path
import pandas as pd

_CANDIDATES = [
    os.environ.get("CAPEVERDE_DATA_DIR"),            # 1. explicit override (env)
    "data",                                          # 2. the bundled verify/data/ inputs
    "../../../../data/cape-verde-diaspora",          # 3. repo-relative to this notebook (phase2/data/...)
    r"D:\AI\journalist agent review\phase2\data\cape-verde-diaspora",  # 4. absolute (this machine)
]
def _resolve_data_dir(cands):
    for c in cands:
        if not c:
            continue
        p = Path(c).expanduser().resolve()
        if (p / "01_diaspora" / "cape_verde_emigrants_by_destination.csv").exists():
            return p
    return None

DATA_DIR = _resolve_data_dir(_CANDIDATES)
assert DATA_DIR is not None, (
    "could not locate the dataset — download the bundled data next to this notebook, "
    "or set CAPEVERDE_DATA_DIR to the data directory, then re-run.")
print("DATA_DIR =", DATA_DIR)

REQUIRED = [
    "01_diaspora/cape_verde_emigrants_by_destination.csv",
    "02_remittances/remittances_pct_gdp_tidy.csv",
    "02_remittances/Metadata_Country_API_BX.TRF.PWKR.DT.GD.ZS_DS2_en_csv_v2_6343.csv",
    "03_football/international_results.csv",
    "03_football/squad_2026.csv",
    "04_population/population_total_tidy.csv",
    "04_population/Metadata_Country_API_SP.POP.TOTL_DS2_EN_csv_v2_3107.csv",
]
for rel in REQUIRED:
    assert (DATA_DIR / rel).exists(), f"missing required input: {rel}"
print("all", len(REQUIRED), "required inputs present")

REPRO_OUT = Path("_repro_out").resolve()
REPRO_OUT.mkdir(exist_ok=True)  # regenerated artifacts only — never write into DATA_DIR


In [ ]:
# cell_diaspora — re-expresses code/diaspora_analysis.py (ana_01..ana_05)
m = pd.read_csv(DATA_DIR / "01_diaspora" / "cape_verde_emigrants_by_destination.csv")
m["destination"] = m["destination"].str.replace("*", "", regex=False).str.strip()
w = m.pivot_table(index="destination", columns="year", values="migrants", aggfunc="sum")
world = w.loc["World"]
named = w.drop(index="World")

# ana_01 — the 2024 floor is literally the sum of the named destinations
# (the union of all 8 waves carries 25 named rows; Estonia and Latvia are zero by
#  2024, so the 2024 matrix names 24 destinations — the published count)
total24, sum24 = int(world[2024]), int(named[2024].sum())
named24 = int((named[2024] > 0).sum())
pt, fr = int(named.loc["Portugal", 2024]), int(named.loc["France", 2024])
print(f"World total (floor) 2024: {total24:,}; sum of ALL named destination rows: {sum24:,} "
      f"({named24} with a non-zero 2024 stock)")
print(f"Portugal {pt:,} ({pt/total24*100:.1f}%) | France {fr:,} ({fr/total24*100:.1f}%) | "
      f"together {(pt+fr)/total24*100:.1f}%")
assert total24 == 146396 and sum24 == total24
assert round(pt/total24*100, 1) == 52.5 and round(fr/total24*100, 1) == 23.8
assert round((pt+fr)/total24*100, 1) == 76.3
for host in ("Netherlands", "United States", "Spain", "Senegal"):
    assert host not in named.index, host + " unexpectedly has a row"
print("OK ana_01: 146,396 = sum of 24 named destinations; NL/US/Spain/Senegal absent")

# ana_02 — growth multiples
mult_world = world[2024] / world[1990]
lux90, lux24 = named.loc["Luxembourg", 1990], named.loc["Luxembourg", 2024]
assert round(mult_world, 2) == 2.83
assert round(lux24 / lux90, 2) == 7.15
assert round(named.loc["France", 2024] / named.loc["France", 1990], 2) == 4.55
print(f"OK ana_02: world x{mult_world:.2f}; Luxembourg x{lux24/lux90:.2f}; France x4.55")

# ana_03 — the Sao Tome decline
stp = w.loc["Sao Tome and Principe"].sort_index()
assert int(stp[1990]) == 4250 and int(stp[2024]) == 1115
assert all(stp.iloc[i] > stp.iloc[i+1] for i in range(len(stp)-1))
assert round((stp[2024]-stp[1990])/stp[1990]*100, 1) == -73.8
print(f"OK ana_03: Sao Tome 4,250 -> 1,115 (-73.8%), declining in every wave")

# ana_04 — the floor as a share of residents
pop = pd.read_csv(DATA_DIR / "04_population" / "population_total_tidy.csv")
cpv_pop = pop[pop["Country Code"] == "CPV"].set_index("year").population
ratio90 = world[1990] / cpv_pop[1990] * 100
ratio24 = world[2024] / cpv_pop[2024] * 100
print(f"1990: {world[1990]:,.0f}/{cpv_pop[1990]:,.0f} = {ratio90:.1f} per 100 | "
      f"2024: {world[2024]:,.0f}/{cpv_pop[2024]:,.0f} = {ratio24:.1f} per 100")
assert round(ratio90, 1) == 13.8 and round(ratio24, 1) == 27.9
print("OK ana_04: the floor ratio doubled, 13.8 -> 27.9 per 100 residents")

# ana_05 — Cabo Verde's ratio (the full 212-origin ranking needs the UN xlsx matrix:
# code/diaspora_analysis.py lines 86-158 reproduces rank #20 of 212 WB-covered origins)
assert round(146396 / 524877 * 100, 1) == 27.9
print("OK ana_05 (ratio): 146,396 / 524,877 = 27.9% — published rank #20 of 212 (see code/diaspora_analysis.py)")


In [ ]:
# cell_remittances — re-expresses code/remittances_analysis.py (ana_06..ana_08)
r = pd.read_csv(DATA_DIR / "02_remittances" / "remittances_pct_gdp_tidy.csv")
meta = pd.read_csv(DATA_DIR / "02_remittances" /
                   "Metadata_Country_API_BX.TRF.PWKR.DT.GD.ZS_DS2_en_csv_v2_6343.csv",
                   encoding="utf-8-sig")
countries = set(meta[meta.Region.notna()]["Country Code"])   # aggregates out
rc = r[r["Country Code"].isin(countries)].copy()
cpv = rc[rc["Country Code"] == "CPV"].sort_values("year")

# ana_06 — the arc
d = dict(zip(cpv.year, cpv.remittances_pct_gdp.round(1)))
assert d[1980] == 28.2 and d[2010] == 7.2 and d[2021] == 15.3 and d[2024] == 12.3
assert cpv.remittances_pct_gdp.round(1).max() == 28.2
print("OK ana_06: 28.2% (1980, peak) -> 7.2% (2010, low) -> 15.3% (2021 surge) -> 12.3% (2024)")

# ana_07/ana_08 — rank history
hist = []
for y in range(1980, 2025):
    yy = rc[rc.year == y].dropna(subset=["remittances_pct_gdp"])
    yy = yy.sort_values("remittances_pct_gdp", ascending=False).reset_index(drop=True)
    row = yy[yy["Country Code"] == "CPV"]
    if row.empty:
        continue
    hist.append((y, int(row.index[0]) + 1, len(yy)))
h = pd.DataFrame(hist, columns=["year", "rank", "n"]).set_index("year")
assert (h.loc[1980, "rank"], h.loc[1980, "n"]) == (2, 90)
assert (h.loc[2010, "rank"], h.loc[2010, "n"]) == (40, 179)
assert (h.loc[2024, "rank"], h.loc[2024, "n"]) == (18, 160)
top10 = h[h["rank"] <= 10]
assert len(top10) == 21 and list(top10.index) == list(range(1980, 2001))
print(f"OK ana_07/ana_08: #2 of 90 (1980); 21 consecutive top-10 years (1980-2000); "
      f"#40 of 179 (2010); #18 of {h.loc[2024,'n']} (2024)")


In [ ]:
# cell_squad — re-expresses code/squad_analysis.py (ana_09..ana_11)
s = pd.read_csv(DATA_DIR / "03_football" / "squad_2026.csv")
n, abroad = len(s), s[s.born_abroad]
assert (n, len(abroad)) == (26, 15) and round(len(abroad)/n*100, 1) == 57.7
bc = s.birth_country.value_counts()
assert bc["Netherlands"] == 6 and bc["Portugal"] == 4 and bc["France"] == 3
print("OK ana_09: 15/26 born abroad (57.7%): NED 6, POR 4, FRA 3, IRL 1, USA 1")

tot_caps, tot_goals = int(s.caps.sum()), int(s.goals.sum())
ab_caps, ab_goals = int(abroad.caps.sum()), int(abroad.goals.sum())
assert (tot_caps, ab_caps) == (823, 414) and round(ab_caps/tot_caps*100, 1) == 50.3
assert (tot_goals, ab_goals) == (75, 33) and round(ab_goals/tot_goals*100, 1) == 44.0
wc_scorers = ["Kevin Pina", "Hélio Varela", "Deroy Duarte", "Sidny Lopes Cabral"]
n_abroad_goals = sum(bool(s[s.name == nm].iloc[0].born_abroad) for nm in wc_scorers)
assert n_abroad_goals == 3
assert int(s[s.name == "Deroy Duarte"].iloc[0].goals) == 0   # his 59' vs Argentina was his first
print("OK ana_10: 414/823 caps (50.3%), 33/75 goals (44.0%); 3 of 4 WC goals from abroad-born")

assert int((s.club_country == "CPV").sum()) == 0 and s.club_country.nunique() == 14
print("OK ana_11: zero of the 26 play club football in Cape Verde; 14 club countries")


In [ ]:
# cell_elo — re-expresses code/elo_ratings.py (ana_12 / ana_14 / ana_15 / ana_16)
CONTINENTAL_FINALS = {"African Cup of Nations", "Copa América", "UEFA Euro", "AFC Asian Cup",
                      "Gold Cup", "CONCACAF Championship", "Oceania Nations Cup", "Confederations Cup"}
NATIONS_LEAGUES = {"UEFA Nations League", "CONCACAF Nations League"}
def k_factor(t):
    if t == "FIFA World Cup": return 60
    if "qualification" in t or t in NATIONS_LEAGUES: return 40
    if t in CONTINENTAL_FINALS: return 50
    if t == "Friendly": return 20
    return 30
def goal_mult(margin):
    return 1.0 if margin <= 1 else (1.5 if margin == 2 else (11 + margin) / 8)

f = pd.read_csv(DATA_DIR / "03_football" / "international_results.csv")
f = f.dropna(subset=["home_score", "away_score"]).sort_values("date", kind="mergesort").reset_index(drop=True)
print(f"played internationals: {len(f):,}")
wcq = f[f.tournament.isin(["FIFA World Cup", "FIFA World Cup qualification"])]
FIFA = set(wcq.home_team) | set(wcq.away_team)

R, N, LAST = {}, {}, {}
pre_h, pre_a, post_h, post_a, n_h, n_a = [], [], [], [], [], []
snap_1998 = None
rows_iter = list(f.itertuples(index=False))
for i, row in enumerate(rows_iter):
    h, a = row.home_team, row.away_team
    rh, ra = R.get(h, 1500.0), R.get(a, 1500.0)
    pre_h.append(rh); pre_a.append(ra); n_h.append(N.get(h, 0)); n_a.append(N.get(a, 0))
    adv = 0 if row.neutral else 100
    we_h = 1 / (10 ** (-((rh + adv) - ra) / 400) + 1)
    margin = abs(row.home_score - row.away_score)
    w_h = 1.0 if row.home_score > row.away_score else (0.0 if row.home_score < row.away_score else 0.5)
    delta = k_factor(row.tournament) * goal_mult(margin) * (w_h - we_h)
    R[h], R[a] = rh + delta, ra - delta
    N[h] = N.get(h, 0) + 1; N[a] = N.get(a, 0) + 1
    LAST[h] = row.date; LAST[a] = row.date
    post_h.append(R[h]); post_a.append(R[a])
    nxt = rows_iter[i + 1].date if i + 1 < len(rows_iter) else "9999"
    if row.date <= "1998-12-31" < nxt:
        snap_1998 = (dict(R), dict(N), dict(LAST))
for c, v in (("pre_home", pre_h), ("pre_away", pre_a), ("post_home", post_h),
             ("post_away", post_a), ("n_home", n_h), ("n_away", n_a)):
    f[c] = v

CV = "Cape Verde"
is_h = f.home_team == CV
cv = f[is_h | (f.away_team == CV)].copy()
cv["cv_pre"] = f.pre_home.where(is_h, f.pre_away)
cv["cv_post"] = f.post_home.where(is_h, f.post_away)
cv["opp"] = f.away_team.where(is_h, f.home_team)
cv["opp_pre"] = f.pre_away.where(is_h, f.pre_home)
cv["cv_score"] = f.home_score.where(is_h, f.away_score)
cv["opp_score"] = f.away_score.where(is_h, f.home_score)

# ana_12 — peak 1702 mid-World-Cup; 1998 bottom #138 of 198; final #64
peak = cv.loc[cv.cv_post.idxmax()]
assert round(peak.cv_post) == 1702 and peak.date == "2026-06-26"
assert round(cv.iloc[-1].cv_post) == 1699
R98, N98, LAST98 = snap_1998
pool98 = {t: r for t, r in R98.items() if N98[t] >= 10 and LAST98[t] >= "1995-01-01" and t in FIFA}
rank98 = 1 + sum(r > R98[CV] for r in pool98.values())
assert (rank98, len(pool98)) == (138, 198) and round(R98[CV]) == 1384
print(f"OK ana_12: 1998 bottom 1384 = #{rank98} of {len(pool98)}; peak {peak.cv_post:.0f} on {peak.date}; "
      f"final {cv.iloc[-1].cv_post:.0f}")

# ana_14 — the four WC2026 matches
wc26 = cv[cv.tournament == "FIFA World Cup"]
exp = [round(1 / (10 ** (-(r.cv_pre - r.opp_pre) / 400) + 1) * 1000) / 10 for r in wc26.itertuples(index=False)]
assert exp == [3.7, 17.1, 49.2, 4.7], exp
assert [f"{r.cv_post - r.cv_pre:+.0f}" for r in wc26.itertuples(index=False)] == ["+28", "+20", "+0", "-3"]
assert f"{wc26.iloc[-1].cv_post - wc26.iloc[0].cv_pre:+.0f}" == "+45"
print("OK ana_14: expectancies 3.7 / 17.1 / 49.2 / 4.7; rating changes +28/+20/+0/-3; net +45")

# ana_15 — the upset ladder
e = f[(f.n_home >= 30) & (f.n_away >= 30)].copy()
e = e[e.home_score != e.away_score].copy()
home_won = e.home_score > e.away_score
adv_pts = (~e.neutral) * 100
e["winner"] = e.home_team.where(home_won, e.away_team)
e["gap"] = ((e.pre_away + 0) - (e.pre_home + adv_pts)).where(home_won,
           (e.pre_home + adv_pts) - (e.pre_away + 0))
cv_ups = e[(e.winner == CV) & (e.gap > 0)].sort_values("gap", ascending=False)
pt_win = cv_ups.iloc[0]
assert pt_win.date == "2015-03-31" and round(pt_win.gap) == 440
rank_440 = int((e.gap > pt_win.gap).sum()) + 1
assert rank_440 == 73 and len(e) == 33081
arg = cv[cv.date == "2026-07-03"].iloc[0]
gap_arg = arg.opp_pre - arg.cv_pre
hyp_rank = int((e.gap > gap_arg).sum()) + 1
hyp_wc = int((e[e.tournament == "FIFA World Cup"].gap > gap_arg).sum()) + 1
assert round(gap_arg) == 523 and hyp_rank == 25 and hyp_wc == 2
print(f"OK ana_15: 2015 Portugal win gap {pt_win.gap:.0f} = #{rank_440} of {len(e):,}; "
      f"beating Argentina (gap {gap_arg:.0f}) would rank ~#{hyp_rank} all-time, #{hyp_wc} among WC finals upsets")

# ana_16 — least-populous team in the top 64
pool = {t: r for t, r in R.items() if N[t] >= 30 and LAST[t] >= "2024-01-01" and t in FIFA}
board = sorted(pool.items(), key=lambda kv: -kv[1])
cv_rank = [t for t, _ in board].index(CV) + 1
assert cv_rank == 64 and len(board) == 210 and round(R[CV]) == 1699
popm = pd.read_csv(DATA_DIR / "04_population" / "Metadata_Country_API_SP.POP.TOTL_DS2_EN_csv_v2_3107.csv",
                   encoding="utf-8-sig")
pop25 = pop[(pop.year == 2025) & (pop["Country Code"].isin(set(popm[popm.Region.notna()]["Country Code"])))]
by_name = pop25.set_index("Country Name").population
TEAM_TO_WB = {"United States": "United States", "South Korea": "Korea, Rep.", "Iran": "Iran, Islamic Rep.",
              "Ivory Coast": "Cote d'Ivoire", "Egypt": "Egypt, Arab Rep.", "Turkey": "Turkiye",
              "Russia": "Russian Federation", "Czech Republic": "Czechia", "Venezuela": "Venezuela, RB",
              "DR Congo": "Congo, Dem. Rep.", "Republic of Ireland": "Ireland", "Cape Verde": "Cabo Verde",
              "Slovakia": "Slovak Republic"}
above = []
for i, (t, r_) in enumerate(board[:cv_rank - 1], 1):
    wbn = TEAM_TO_WB.get(t, t)
    if wbn in by_name.index:
        above.append((t, int(by_name[wbn])))
min_above = min(above, key=lambda x: x[1])
cv_pop = int(by_name["Cabo Verde"])
assert min_above[0] == "Kosovo" and min_above[1] == 1576876 and cv_pop == 527326
assert round(cv_pop / R[CV]) == 310
print(f"OK ana_16: Cape Verde #64 of 210 at {R[CV]:.0f}; smallest above = Kosovo (1,576,876); "
      f"310 people per rating point")


In [ ]:
# cell_outcome_model — ana_19: the published ordered-logit constants applied to the
# reproduced pre-match ratings (the FIT itself is reproduced by code/fit_outcome_model.py).
import math
C, B, SCALE = 0.6614, 2.0468, 400.0
def sig(z): return 1.0 / (1.0 + math.exp(-z))
def r1(x): return math.floor(x * 10 + 0.5) / 10
arg = cv[cv.date == "2026-07-03"].iloc[0]
print(f"reproduced pre-match ratings 2026-07-03: Argentina {arg.opp_pre:.1f}, Cape Verde {arg.cv_pre:.1f}")
assert round(arg.cv_pre, 1) == 1701.6 and round(arg.opp_pre, 1) == 2225.0
u = B * ((arg.cv_pre - arg.opp_pre) / SCALE)          # neutral venue
sHi, sLo = sig(C - u), sig(-C - u)
win, draw, loss = r1((1 - sHi) * 100), r1((sHi - sLo) * 100), r1(sLo * 100)
advp = r1((1 - sHi + 0.5 * (sHi - sLo)) * 100)
print(f"model, from Cape Verde's side: win {win}%, level after 90' {draw}%, lose {loss}%; advance {advp}%")
assert (win, draw, loss, advp) == (3.4, 8.3, 88.3, 7.6)
print("OK ana_19: 3.4 / 8.3 / 88.3, advance 7.6 — reproduced from raw data + published constants")


In [ ]:
# cell_wc_population — re-expresses code/wc_smallest_nations.py (ana_17 / ana_18)
TEAM_TO_WB2 = {"United States": "United States", "South Korea": "Korea, Rep.",
               "North Korea": "Korea, Dem. People's Rep.", "Iran": "Iran, Islamic Rep.",
               "Ivory Coast": "Cote d'Ivoire", "Egypt": "Egypt, Arab Rep.", "Turkey": "Turkiye",
               "Russia": "Russian Federation", "Czech Republic": "Czechia", "Venezuela": "Venezuela, RB",
               "DR Congo": "Congo, Dem. Rep.", "Republic of Ireland": "Ireland", "Cape Verde": "Cabo Verde",
               "Curaçao": "Curacao", "Slovakia": "Slovak Republic"}
NO_WB = {"England", "Scotland", "Wales", "Northern Ireland", "Czechoslovakia", "Yugoslavia", "German DR"}
wc = f[f.tournament == "FIFA World Cup"].copy()
wc["year"] = wc.date.str[:4].astype(int)
pop_lut = pop.set_index(["Country Name", "year"]).population
long = pd.concat([wc.rename(columns={"home_team": "team"})[["year", "team"]],
                  wc.rename(columns={"away_team": "team"})[["year", "team"]]])
counts = long.groupby(["year", "team"]).size().rename("matches").reset_index()
rows = []
for r_ in counts.itertuples(index=False):
    if r_.year < 1962 or r_.team in NO_WB:
        continue
    key = (TEAM_TO_WB2.get(r_.team, r_.team), min(r_.year, 2025))
    assert key in pop_lut.index, f"unmapped participant: {r_.team} ({r_.year})"
    rows.append((r_.year, r_.team, int(pop_lut[key]), int(r_.matches), r_.matches >= 4))
d = pd.DataFrame(rows, columns=["edition", "team", "population", "matches", "advanced"])
print(f"participant-edition rows: {len(d)}")

# ana_17 — third-least-populous participant ever
cv_row = d[(d.team == "Cape Verde") & (d.edition == 2026)].iloc[0]
smaller = d[d.population < cv_row.population]
assert len(smaller) == 2
assert set(zip(smaller.team, smaller.edition)) == {("Curaçao", 2026), ("Iceland", 2018)}
assert set(smaller.population) == {156263, 352721} and cv_row.population == 527326
sub1m = d[d.population < 1_000_000]
assert len(sub1m) == 3 and set(sub1m.edition) == {2018, 2026}
print("OK ana_17: only Curaçao 2026 (156,263) and Iceland 2018 (352,721) were smaller; "
      "all three sub-million participants since 2018")

# ana_18 — smallest advancer ever, by 5.1x
adv = d[d.advanced].sort_values("population").reset_index(drop=True)
assert adv.iloc[0].team == "Cape Verde"
nxt = adv.iloc[1]
assert (nxt.team, nxt.edition, int(nxt.population)) == ("Uruguay", 1966, 2707646)
assert round(nxt.population / cv_row.population, 1) == 5.1
p26 = d[d.edition == 2026]
assert round(p26.population.mean() / cv_row.population) == 89
big = p26.sort_values("population", ascending=False).iloc[0]
assert big.team == "United States" and round(big.population / cv_row.population) == 648
assert not smaller.advanced.any()   # Curaçao + Iceland both exited after 3 group games
print(f"OK ana_18: smallest advancer ever; next = Uruguay 1966 at 5.1x; "
      f"mean 2026 participant {p26.population.mean():,.0f} = 89x; US = 648x")


In [ ]:
# cell_findings — the published headline numbers, all reproduced above
print("""REPRODUCED AND ASSERTED (finding -> cell -> script -> data):
  146,396 UN floor = sum of 24 named destinations; PT 52.5% / FR 23.8% (ana_01 -> cell_diaspora -> diaspora_analysis.py -> 01_diaspora)
  world x2.83, Luxembourg x7.15, France x4.55 (ana_02); Sao Tome 4,250->1,115, -73.8% (ana_03)
  floor ratio 13.8 -> 27.9 per 100 residents (ana_04); 27.9% ratio behind rank #20 of 212 (ana_05)
  remittances 28.2% (1980) / 7.2% (2010) / 15.3% (2021) / 12.3% (2024) (ana_06 -> cell_remittances)
  rank #2 of 90 (1980), 21 straight top-10 years, #40 (2010), #18 of 160 (2024) (ana_07/ana_08)
  squad 15/26 = 57.7% born abroad; 414/823 caps = 50.3%; 33/75 goals = 44.0%; 3 of 4 WC goals; 0 domestic (ana_09/10/11 -> cell_squad)
  Elo: 1998 bottom 1384 = #138 of 198; peak 1702 on 2026-06-26; final 1699 = #64 of 210 (ana_12 -> cell_elo)
  WC arc expectancies 3.7/17.1/49.2/4.7, net +45 (ana_14); Portugal-2015 gap 440 = #73 of 33,081; Argentina hypothetical ~#25 / #2 (ana_15)
  least-populous in the top 64; Kosovo min-above at 1,576,876; 310 people per Elo point (ana_16)
  odds 3.4 / 8.3 / 88.3, advance 7.6 (ana_19 -> cell_outcome_model)
  3rd-least-populous WC nation ever; smallest advancer by 5.1x; mean 2026 participant 89x; US 648x (ana_17/ana_18 -> cell_wc_population)
All asserts passed — every published headline number reproduces from the raw dataset.""")


## Provenance & licenses

| Input | Source | License |
|---|---|---|
| `01_diaspora/` | UN DESA International Migrant Stock 2024 (bilateral matrix) | UN data, © UN — cite source |
| `02_remittances/` | World Bank WDI `BX.TRF.PWKR.DT.GD.ZS` | CC BY 4.0 |
| `03_football/international_results.csv` | martj42/international_results | CC0 |
| `03_football/squad_2026.csv` | 2026 FIFA World Cup squads (Wikipedia, FCF registration) | CC BY-SA |
| `04_population/` | World Bank WDI `SP.POP.TOTL` | CC BY 4.0 |

Conventions asserted throughout: football scores are extra-time-inclusive with
shootouts recorded as draws; the 8 unplayed R16 fixtures (NaN scores) are dropped;
every number is "as of 2026-07-03"; the Elo ratings are OUR documented implementation
(start 1500, K 60/50/40/30/20, goal-difference multiplier, +100 home advantage) and are
not comparable with published Elo boards; the UN diaspora numbers are the
born-in-Cabo-Verde FLOOR layer, never mixed with census or heritage estimates.
